# TM account analysis (OFS) — checks before use

Three checks on `cdm_tm_ofs_bal_analysis_fact_v` before it feeds attrition, value and liquidity work.

Cells
1. Setup and config
2. Snapshots and month label — monthly partitions (latest load per month kept); per snapshot the analysis month (`curnt_mnth_nbr`) and the latest real close date; then the account balances compared with daily balances of the same calendar month and of the month before, to see which month a snapshot describes
3. Account key — which column set is one row per account in the latest snapshot, what differs between rows that share an account number, and whether `acct_dim_key` stays the same for an account from one month to the next
4. New and closed accounts per month — accounts entering and leaving the analysis, how many come back within a few months, how many leavers carry a close date, balances leaving; the same for customers (power id, all accounts gone); split DAP / CAA; how close dates relate to the analysis month

Uses only columns seen in the profile of 2026-11-08. Tables are saved as CSV in OUT_DIR.

In [ ]:
# ============================================================
# 1. Setup and config
# ============================================================
import os, re, html, warnings, datetime as _dt
warnings.filterwarnings("ignore", category=ResourceWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore")

# --- NumPy alias shim (old pyspark code paths use removed aliases) ---
import numpy as np
for _a, _t in (("object0", object), ("bool8", np.bool_), ("int0", np.intp), ("uint0", np.uintp),
               ("bool", bool), ("object", object)):
    if not hasattr(np, _a):
        setattr(np, _a, _t)

# --- pandas shim: PySpark 3.3 createDataFrame calls iteritems (removed in pandas 2.0) ---
import pandas as pd
if not hasattr(pd.DataFrame, "iteritems"):
    pd.DataFrame.iteritems = pd.DataFrame.items
if not hasattr(pd.Series, "iteritems"):
    pd.Series.iteritems = pd.Series.items

from IPython.display import display, HTML
from pyspark.sql import SparkSession, Window, functions as F, types as T
from pyspark import StorageLevel

spark = (SparkSession.builder
         .appName("cdm_tm_ofs_checks")
         .config("spark.ui.showConsoleProgress", "false")   # no stage progress bars in cell outputs
         .enableHiveSupport()
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
try:  # silence Atlas lineage hook + harmless executor-release noise (log4j2 on Spark 3.3)
    _L = spark._jvm.org.apache.logging.log4j
    for n in ("org.apache.atlas", "com.hortonworks.spark.atlas", "com.cloudera.spark.lineage",
              "org.apache.kafka", "org.apache.spark.sql.catalyst.util.package",
              "org.apache.spark.network", "io.netty"):
        _L.core.config.Configurator.setLevel(n, _L.Level.OFF)
except Exception:
    pass
# Unparseable date strings -> NULL instead of SparkUpgradeException (we probe many formats)
spark.conf.set("spark.sql.legacy.timeParserPolicy", "CORRECTED")

# ---------------- config ----------------
OFS             = "clhhd01p_clh.cdm_tm_ofs_bal_analysis_fact_v"    # monthly TM account analysis, partitioned by time_key
DEP             = "clhhd01p_clh.tap_deposit_balance_fact_daily_v"  # daily balances (~6 months), used to test the month label
N_MONTHS        = 27      # latest monthly partitions used: 24 month-to-month transitions + 3 months to see re-entries
EXIT_LOOKAHEAD  = 3       # an exit counts as "back" if the account reappears within this many missing months
ALIGN_SNAPS     = 3       # latest OFS months compared with daily balances
OUT_DIR         = "../explore/cdm_tm_ofs_checks"
os.makedirs(OUT_DIR, exist_ok=True)

# ---------------- helpers ----------------
CSS = """<style>
.cx {font-family: Arial, sans-serif; font-size: 12px;}
.cx h3 {margin: 14px 0 4px 0; font-size: 14px;}
.cx h4 {margin: 10px 0 3px 0; font-size: 13px; color: #1f3b63;}
.cx table {border-collapse: collapse; margin: 2px 0 8px 0;}
.cx th {background: #e8eef6; border: 1px solid #c8d2e0; padding: 2px 6px; text-align: left; white-space: nowrap;}
.cx td {border: 1px solid #dde3ec; padding: 2px 6px; vertical-align: top; max-width: 420px; overflow-wrap: anywhere;}
.cx .note {color: #555; margin: 2px 0;}
.cx .warn {color: #a33; font-weight: bold;}
.cx pre {background: #f6f8fa; border: 1px solid #dde3ec; padding: 6px; white-space: pre-wrap; font-size: 11px; max-width: 1400px;}
</style>"""

def esc(x):
    return html.escape("" if x is None else str(x))

def tbl(pdf, max_rows=None):
    """pandas DataFrame -> compact HTML table (values escaped)."""
    if pdf is None or len(pdf) == 0:
        return "<p class='note'>(no rows)</p>"
    if max_rows is not None:
        pdf = pdf.head(max_rows)
    head = "".join(f"<th>{esc(c)}</th>" for c in pdf.columns)
    body = "".join("<tr>" + "".join(f"<td>{esc(v)}</td>" for v in row) + "</tr>"
                   for row in pdf.astype(object).where(pdf.notna(), "").itertuples(index=False))
    return f"<table><tr>{head}</tr>{body}</table>"

def show(parts):
    """One consolidated HTML output per cell; tex2jax_ignore keeps any dollar sign from turning into math."""
    display(HTML(CSS + "<div class='cx tex2jax_ignore mathjax_ignore'>" + "".join(parts) + "</div>"))

def fnum(x):
    if x is None:
        return ""
    try:
        x = float(x)
    except Exception:
        return str(x)
    if np.isnan(x):
        return ""
    a = abs(x)
    for d, s in ((1e12, "T"), (1e9, "B"), (1e6, "M"), (1e3, "K")):
        if a >= d:
            return f"{x / d:.2f}{s}"
    if x == int(x):
        return str(int(x))
    return f"{x:.4g}"

def fpct(x, digits=1):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return ""
    return f"{100 * float(x):.{digits}f}%"

def qcol(c):
    return F.col(f"`{c}`")

NUMERIC_T = (T.ByteType, T.ShortType, T.IntegerType, T.LongType, T.FloatType, T.DoubleType, T.DecimalType)
COMPLEX_T = (T.ArrayType, T.MapType, T.StructType, T.BinaryType)
DATE_NAME = re.compile(r"(_dt$|_dte$|_date$|^date_|_date_|_ts$|_tmstmp|timestamp|^dt_|_dt_|asof|as_of|snap|_month$|_mo$|_yrmo|yyyymm|period)")

def kind_of(dtype):
    if isinstance(dtype, (T.DateType, T.TimestampType)): return "date"
    if isinstance(dtype, T.BooleanType): return "bool"
    if isinstance(dtype, NUMERIC_T): return "num"
    if isinstance(dtype, COMPLEX_T): return "complex"
    return "str"

def is_date_like(name, dtype):
    k = kind_of(dtype)
    if k == "date":
        return True
    return k in ("str", "num") and bool(DATE_NAME.search(name.lower())) \
        and not isinstance(dtype, (T.FloatType, T.DoubleType))

def date_expr(c, dtype):
    """Parse a date-like column of any common storage format to DATE (NULL if unparseable)."""
    col = qcol(c)
    if isinstance(dtype, T.DateType):
        return col
    if isinstance(dtype, T.TimestampType):
        return F.to_date(col)
    s = F.trim(col.cast("string"))
    return (F.when(s.rlike(r"^\d{4}-\d{2}-\d{2}"), F.to_date(F.substring(s, 1, 10), "yyyy-MM-dd"))
             .when(s.rlike(r"^\d{8}$"), F.to_date(s, "yyyyMMdd"))
             .when(s.rlike(r"^\d{1,2}/\d{1,2}/\d{4}"),
                   F.to_date(F.regexp_extract(s, r"^(\d{1,2}/\d{1,2}/\d{4})", 1), "M/d/yyyy"))
             .when(s.rlike(r"^\d{6}$"), F.to_date(F.concat(s, F.lit("01")), "yyyyMMdd"))
             .when(s.rlike(r"^\d{4}-\d{2}$"), F.to_date(F.concat(s, F.lit("-01")), "yyyy-MM-dd")))

def real_date(p):
    """Drop placeholder dates (year <= 1900 or >= 2100, e.g. 0001-01-01 / 9999-12-31) for min/max."""
    return F.when(F.year(p).between(1901, 2099), p)

show([f"<h3>Setup</h3><p>Spark {esc(spark.version)} · pandas {esc(pd.__version__)} · numpy {esc(np.__version__)}</p>",
      tbl(pd.DataFrame([("OFS", OFS), ("DEP", DEP), ("N_MONTHS", N_MONTHS), ("EXIT_LOOKAHEAD", EXIT_LOOKAHEAD),
                        ("ALIGN_SNAPS", ALIGN_SNAPS), ("OUT_DIR", os.path.abspath(OUT_DIR))],
                       columns=["setting", "value"]))])


In [ ]:
# ============================================================
# 2. Snapshots and month label
# ============================================================
ofs = spark.table(OFS)
need = ["time_key", "ofsa_as_of_dt", "curnt_mnth_nbr", "bank", "appl_id", "src_acct_no", "data_src_cd",
        "acct_dim_key", "seq_id", "cust_pwr_id", "acct_close_dt", "avg_investable_bal", "cor_tda_avg_bal",
        "agg_ldgr_bal_curcycle", "cycle_days"]
miss = [c for c in need if c not in ofs.columns]
if miss:
    raise RuntimeError(f"{OFS} lacks columns {miss}")

def real_dt(c):
    """String / yyyymmdd date -> DATE; placeholders (9999-12-31, 0, blank) -> NULL."""
    s = F.trim(F.col(c).cast("string"))
    d = (F.when(s.rlike(r"^\d{4}-\d{2}-\d{2}"), F.to_date(F.substring(s, 1, 10), "yyyy-MM-dd"))
          .when(s.rlike(r"^\d{8}$"), F.to_date(s, "yyyyMMdd")))
    return F.when(F.year(d).between(1901, 2099), d)

def prev_ym(ym):
    y, m = divmod(int(ym), 100)
    return (y - 1) * 100 + 12 if m == 1 else ym - 1

def ym_label(ym):
    return f"{int(ym) // 100}-{int(ym) % 100:02d}"

# Monthly partitions (time_key = yyyymmdd); several loads in one month -> keep the latest
try:
    _pp = spark.sql(f"SHOW PARTITIONS {OFS}").toPandas().iloc[:, 0].astype(str)
    tks = sorted({int(v.split("time_key=", 1)[1].split("/")[0]) for v in _pp if "time_key=" in v})
except Exception:
    tks = []
if not tks:
    tks = sorted(int(r[0]) for r in ofs.select("time_key").distinct().collect() if r[0] is not None)
tks = [t for t in tks if 19000101 <= t <= 21001231]
by_month = {}
for t in tks:
    by_month[t // 100] = max(by_month.get(t // 100, 0), t)
n_extra = len(tks) - len(by_month)
TKS = sorted(by_month.values())[-N_MONTHS:]
if len(TKS) < 3:
    raise RuntimeError(f"Only {len(TKS)} monthly partitions found in {OFS}")
IDX = {t: i for i, t in enumerate(TKS)}
LAST = len(TKS) - 1
base = ofs.filter(F.col("time_key").isin(TKS))

# Analysis month per snapshot
cm = base.groupBy("time_key", F.col("curnt_mnth_nbr").cast("int").alias("cmn")).count().toPandas()
od = (base.groupBy("time_key")
          .agg(F.count(F.lit(1)).alias("rows"), F.max("ofsa_as_of_dt").alias("ofsa_as_of_dt"),
               F.countDistinct("ofsa_as_of_dt").alias("n_ofsa"), F.max(real_dt("acct_close_dt")).alias("max_close"))
          .toPandas().set_index("time_key"))
arows = []
for t in TKS:
    g = cm[cm["time_key"] == t].sort_values("count", ascending=False)
    top = g.iloc[0]
    m = (t // 100) % 100
    sh = None if pd.isna(top["cmn"]) else int((m - int(top["cmn"])) % 12)
    o = od.loc[t]
    arows.append({"time_key": t, "ofsa_as_of_dt": o["ofsa_as_of_dt"] + ("" if o["n_ofsa"] == 1 else f" (+{o['n_ofsa'] - 1})"),
                  "rows": f"{int(o['rows']):,}", "curnt_mnth_nbr (most rows)": "" if pd.isna(top["cmn"]) else int(top["cmn"]),
                  "share": fpct(top["count"] / g["count"].sum()),
                  "months behind snapshot": sh, "latest real close date": o["max_close"]})
AL = pd.DataFrame(arows)
SHIFT = int(pd.Series([r for r in AL["months behind snapshot"] if r is not None]).mode().iloc[0])

# Balance test: snapshot balance vs daily balances of the same calendar month and of the month before
bal_parts, BAL_SHIFT = [], None
try:
    dep = spark.table(DEP)
    dmiss = [c for c in ("ldgr_dt_key", "acct_dim_key", "dly_spot_bal") if c not in dep.columns]
except Exception as e:
    dmiss = [f"cannot read: {str(e).splitlines()[0][:150]}"]
if dmiss:
    bal_parts.append(f"<p class='note'>Balance test skipped: {esc(DEP)} {esc(', '.join(dmiss))}</p>")
else:
    rng = dep.agg(F.min("ldgr_dt_key"), F.max("ldgr_dt_key")).collect()[0]
    d_lo = _dt.datetime.strptime(str(rng[0]), "%Y%m%d").date()
    d_hi = _dt.datetime.strptime(str(rng[1]), "%Y%m%d").date()
    def covered(ym):
        s = _dt.date(ym // 100, ym % 100, 1)
        e = (s + _dt.timedelta(days=32)).replace(day=1) - _dt.timedelta(days=1)
        return d_lo <= s + _dt.timedelta(days=4) and d_hi >= e - _dt.timedelta(days=4)
    cands = [t for t in TKS if covered(t // 100) and covered(prev_ym(t // 100))][-ALIGN_SNAPS:]
    if not cands:
        bal_parts.append(f"<p class='note'>Balance test skipped: daily balances ({d_lo} to {d_hi}) do not cover two "
                         "full months before any OFS snapshot.</p>")
    else:
        months = sorted({t // 100 for t in cands} | {prev_ym(t // 100) for t in cands})
        ldg_ym = F.substring(F.col("ldgr_dt_key").cast("string"), 1, 6)
        dm = (dep.filter(ldg_ym.isin([str(x) for x in months]) & (F.col("acct_dim_key") > 0))
                 .groupBy(F.col("acct_dim_key").alias("k"), ldg_ym.cast("int").alias("ym"))
                 .agg(F.avg(F.col("dly_spot_bal").cast("double")).alias("d_avg")))
        o = (base.filter(F.col("time_key").isin(cands) & (F.col("acct_dim_key") > 0))
                 .groupBy("time_key", F.col("acct_dim_key").alias("k"))
                 .agg(F.count(F.lit(1)).alias("n"),
                      F.first(F.col("cor_tda_avg_bal").cast("double")).alias("cor_tda_avg_bal"),
                      F.first(F.when(F.col("cycle_days") > 0, F.col("agg_ldgr_bal_curcycle").cast("double")
                                     / F.col("cycle_days").cast("double"))).alias("ledger_per_cycle_day"))
                 .filter("n = 1"))   # accounts that appear once in the snapshot
        ym0 = (F.col("time_key") / 100).cast("long")
        ym1 = F.when(ym0 % 100 == 1, ym0 - 89).otherwise(ym0 - 1)
        brows = []
        for h, lab in ((0, "same month as snapshot"), (1, "month before snapshot")):
            j = o.withColumn("ym", ym0 if h == 0 else ym1).join(dm, ["k", "ym"])
            for mcol in ("cor_tda_avg_bal", "ledger_per_cycle_day"):
                rd = F.abs(F.col(mcol) - F.col("d_avg")) / F.greatest(F.abs(F.col("d_avg")), F.lit(100.0))
                r = (j.filter(F.col(mcol).isNotNull()).groupBy("time_key")
                      .agg(F.count(F.lit(1)).alias("n"), F.percentile_approx(rd, 0.5, 1000).alias("med"),
                           F.avg((rd <= 0.02).cast("int")).alias("w2"), F.avg((rd <= 0.10).cast("int")).alias("w10"))
                      .toPandas())
                for _, x in r.iterrows():
                    brows.append({"snapshot": int(x["time_key"]), "OFS balance": mcol, "daily balances of": lab,
                                  "h": h, "accounts matched": int(x["n"]), "median difference": fpct(x["med"]),
                                  "within 2%": x["w2"], "within 10%": x["w10"]})
        BR = pd.DataFrame(brows)
        if len(BR):
            sc = BR.groupby("h")["within 10%"].mean()
            BAL_SHIFT = int(sc.idxmax())
            BR = BR.sort_values(["snapshot", "OFS balance", "h"])
            BR["within 2%"] = BR["within 2%"].map(fpct)
            BR["within 10%"] = BR["within 10%"].map(fpct)
            BR["accounts matched"] = BR["accounts matched"].map(lambda v: f"{v:,}")
            bal_parts += [tbl(BR.drop(columns="h")),
                          f"<p>Balances match the <b>{'month before' if BAL_SHIFT == 1 else 'same month as'}</b> the "
                          f"snapshot best (average share within 10%: same month {fpct(sc.get(0))}, month before "
                          f"{fpct(sc.get(1))}).</p>"]
        else:
            bal_parts.append("<p class='note'>No account matched between the two tables on acct_dim_key.</p>")
        bal_parts.append(f"<p class='note'>Daily balances: {esc(DEP)}, mean of dly_spot_bal over business days "
                         f"{d_lo} to {d_hi}; the analysis averages calendar days, so small differences are expected.</p>")

AL.to_csv(os.path.join(OUT_DIR, "snapshots.csv"), index=False)
both = "agree" if BAL_SHIFT is None or BAL_SHIFT == SHIFT else "DISAGREE"
show(["<h3>Snapshots and month label</h3>",
      f"<p>{len(TKS)} monthly snapshots used, {ym_label(TKS[0] // 100)} to {ym_label(TKS[-1] // 100)}"
      + (f"; {n_extra} extra mid-month load(s) in the table ignored (latest per month kept)" if n_extra else "") + ".</p>",
      f"<p>curnt_mnth_nbr says a snapshot holds the analysis of <b>{SHIFT} month(s) before</b> its date"
      + ("" if BAL_SHIFT is None else f"; balances say {BAL_SHIFT}: the two checks <b>{both}</b>") + ".</p>",
      tbl(AL), "<h4>Balance test</h4>"] + bal_parts)


In [ ]:
# ============================================================
# 3. Account key (latest snapshot) and acct_dim_key stability
# ============================================================
lt = base.filter(F.col("time_key") == TKS[-1])
N_LT = lt.count()
CANDS = [["src_acct_no"], ["bank", "src_acct_no"], ["appl_id", "src_acct_no"], ["bank", "appl_id", "src_acct_no"],
         ["data_src_cd", "bank", "appl_id", "src_acct_no"], ["acct_dim_key"], ["seq_id"]]

def nonnull(cols):
    cond = F.lit(True)
    for c in cols:
        cond = cond & F.col(c).isNotNull()
        if c.endswith("_key") or c == "seq_id":
            cond = cond & (F.col(c) > 0)
    return cond

krows, KST = [], {}
for ks in CANDS:
    r = (lt.filter(nonnull(ks)).groupBy(*ks).count()
           .agg(F.count(F.lit(1)).alias("vals"), F.sum("count").alias("rows"),
                F.sum(F.when(F.col("count") > 1, 1).otherwise(0)).alias("dup_vals"),
                F.sum(F.when(F.col("count") > 1, F.col("count")).otherwise(0)).alias("dup_rows"),
                F.max("count").alias("mx")).collect()[0])
    rows_ = r["rows"] or 0
    KST[tuple(ks)] = (rows_, r["dup_vals"] or 0, r["dup_rows"] or 0)
    krows.append({"key": " + ".join(ks), "rows with key": f"{rows_:,} ({fpct(rows_ / N_LT)})",
                  "distinct values": f"{(r['vals'] or 0):,}", "values on 2+ rows": f"{(r['dup_vals'] or 0):,}",
                  "rows involved": f"{(r['dup_rows'] or 0):,}", "max rows per value": r["mx"]})

natural = [tuple(k) for k in CANDS[:5]]
uniq = [k for k in natural if KST[k][1] == 0 and KST[k][0] == N_LT]
KEY = list(uniq[0]) if uniq else list(min(natural, key=lambda k: KST[k][2]))
key_note = (f"Chosen account key: {' + '.join(KEY)} (one row per value, always filled)." if uniq else
            f"No natural key is unique; using {' + '.join(KEY)} (fewest repeated rows); repeats are merged in cell 4.")

# What differs between rows that share an account number
VARY = [c for c in ["bank", "appl_id", "data_src_cd", "src_syst", "acct_dim_key", "seq_id", "hier_level", "cust_pwr_id",
                    "lead_acct_no", "acct_close_dt", "cor_sub_prod", "avg_investable_bal"] if c in lt.columns]
dupk = lt.filter(F.col("src_acct_no").isNotNull()).groupBy("src_acct_no").count().filter("count > 1").select("src_acct_no")
dd = lt.join(dupk, "src_acct_no").persist(StorageLevel.MEMORY_AND_DISK)
va = dd.groupBy("src_acct_no").agg(*[F.countDistinct(F.col(c).cast("string")).alias(c) for c in VARY])
vs = va.agg(F.count(F.lit(1)).alias("groups"), *[F.avg((F.col(c) > 1).cast("int")).alias(c) for c in VARY]).collect()[0]
vary_t = pd.DataFrame([(c, fpct(vs[c])) for c in VARY], columns=["column", "differs within the group"]) \
           .sort_values("differs within the group", key=lambda s: s.str.rstrip("%").astype(float), ascending=False)
ex_keys = dd.groupBy("src_acct_no").count().orderBy(F.col("count").desc(), "src_acct_no").limit(3)
ex = (dd.join(ex_keys.select("src_acct_no"), "src_acct_no")
        .select("src_acct_no", *[F.col(c).cast("string").alias(c) for c in VARY if c != "src_acct_no"])
        .orderBy("src_acct_no", "bank").toPandas())
dd.unpersist()

# acct_dim_key from one month to the next, for accounts present once in both
def once(df):
    return (df.filter(nonnull(KEY)).groupBy(*KEY)
              .agg(F.count(F.lit(1)).alias("n"), F.max("acct_dim_key").alias("adk")).filter("n = 1").drop("n"))
st = (once(lt).alias("a").join(once(base.filter(F.col("time_key") == TKS[-2])).alias("b"), KEY)
        .agg(F.count(F.lit(1)).alias("n"), F.avg((F.col("a.adk") == F.col("b.adk")).cast("int")).alias("same"),
             F.avg(((F.col("a.adk") > 0) & (F.col("b.adk") > 0)).cast("int")).alias("both_set")).collect()[0])

show(["<h3>Account key — snapshot " + esc(TKS[-1]) + f", {N_LT:,} rows</h3>", tbl(pd.DataFrame(krows)),
      f"<p><b>{esc(key_note)}</b></p>",
      f"<h4>Rows sharing an account number: {vs['groups']:,} account numbers — what differs</h4>", tbl(vary_t),
      "<p class='note'>Examples (3 account numbers with the most rows):</p>", tbl(ex),
      "<h4>acct_dim_key over time</h4>",
      tbl(pd.DataFrame([("Accounts in both of the last two snapshots (key above)", f"{st['n']:,}"),
                        ("Same acct_dim_key in both", fpct(st["same"])),
                        ("acct_dim_key set (> 0) in both", fpct(st["both_set"]))], columns=["check", "value"]))])


In [ ]:
# ============================================================
# 4. New and closed accounts per month (and customers)
# ============================================================
L = EXIT_LOOKAHEAD
idx_map = F.create_map(*[x for t in TKS for x in (F.lit(t), F.lit(IDX[t]))])
pwr = F.trim(F.col("cust_pwr_id").cast("string"))
pres = (base.filter(nonnull(KEY))
            .select(*KEY, "time_key", "data_src_cd",
                    F.when(pwr.isNotNull() & ~pwr.isin("", "-1"), pwr).alias("pwr"),
                    F.col("avg_investable_bal").cast("double").alias("inv"), real_dt("acct_close_dt").alias("close"))
            .groupBy(*KEY, "time_key")
            .agg(F.max("data_src_cd").alias("src"), F.max("pwr").alias("pwr"),
                 F.sum("inv").alias("inv"), F.max("close").alias("close"))
            .withColumn("i", idx_map[F.col("time_key")]))
w = Window.partitionBy(*KEY).orderBy("i")
pres = (pres.withColumn("nxt", F.lead("i").over(w)).withColumn("prv", F.lag("i").over(w))
            .persist(StorageLevel.MEMORY_AND_DISK))

def exits(df):
    """Rows present at i and absent at i+1 (i before the last snapshot); back = reappears after <= L missing months."""
    return (df.filter((F.col("i") < LAST) & (F.col("nxt").isNull() | (F.col("nxt") > F.col("i") + 1)))
              .withColumn("e", F.col("i") + 1)
              .withColumn("obs", (F.col("i") + L + 1 <= LAST).cast("int"))
              .withColumn("back", (F.col("nxt").isNotNull() & (F.col("nxt") - F.col("i") - 1 <= L)).cast("int")))

acc = (pres.groupBy("i").agg(F.count(F.lit(1)).alias("accounts"), F.sum("inv").alias("inv"),
                              F.sum(((F.col("i") > 0) & F.col("prv").isNull()).cast("int")).alias("new"),
                              F.sum(((F.col("i") > 0) & (F.col("prv") < F.col("i") - 1)).cast("int")).alias("returned"))
           .toPandas().set_index("i"))
ex = exits(pres)
exm = (ex.groupBy("e").agg(F.count(F.lit(1)).alias("exited"), F.sum("inv").alias("ex_inv"),
                           F.avg(F.col("close").isNotNull().cast("int")).alias("ex_close"),
                           F.sum("obs").alias("obs"), F.sum(F.col("obs") * F.col("back")).alias("back"))
         .toPandas().set_index("e"))

# customers: power id with at least one account; exit = all its accounts gone
wc = Window.partitionBy("pwr").orderBy("i")
cp = (pres.filter(F.col("pwr").isNotNull()).groupBy("pwr", "i").agg(F.sum("inv").alias("inv"))
          .withColumn("nxt", F.lead("i").over(wc)))
cacc = cp.groupBy("i").agg(F.count(F.lit(1)).alias("customers"), F.sum("inv").alias("cinv")).toPandas().set_index("i")
cxm = (exits(cp).groupBy("e").agg(F.count(F.lit(1)).alias("c_exited"), F.sum("inv").alias("c_ex_inv"),
                                  F.sum("obs").alias("c_obs"), F.sum(F.col("obs") * F.col("back")).alias("c_back"))
                .toPandas().set_index("e"))

def g(df, i, c, default=0):
    return df.loc[i, c] if i in df.index and not pd.isna(df.loc[i, c]) else default

def pct(a, b):
    return fpct(a / b) if b else ""

mrows = []
for i in range(1, LAST + 1):
    t = TKS[i]
    cyc = t // 100
    for _ in range(SHIFT):
        cyc = prev_ym(cyc)
    a_prev, ci_prev = g(acc, i - 1, "accounts"), g(cacc, i - 1, "customers")
    mrows.append({"snapshot": t, "analysis month": ym_label(cyc), "accounts": f"{int(g(acc, i, 'accounts')):,}",
                  "new": f"{int(g(acc, i, 'new')):,}", "returned": f"{int(g(acc, i, 'returned')):,}",
                  "left": f"{int(g(exm, i, 'exited')):,}", "left %": pct(g(exm, i, "exited"), a_prev),
                  "balance leaving %": pct(g(exm, i, "ex_inv"), g(acc, i - 1, "inv")),
                  "leavers with close date": fpct(g(exm, i, "ex_close", None)) if g(exm, i, "exited") else "",
                  f"leavers back within {L}": pct(g(exm, i, "back"), g(exm, i, "obs")),
                  "customers": f"{int(g(cacc, i, 'customers')):,}", "customers left": f"{int(g(cxm, i, 'c_exited')):,}",
                  "customers left %": pct(g(cxm, i, "c_exited"), ci_prev),
                  "customer balance leaving %": pct(g(cxm, i, "c_ex_inv"), g(cacc, i - 1, "cinv")),
                  f"customers back within {L}": pct(g(cxm, i, "c_back"), g(cxm, i, "c_obs"))})
MT = pd.DataFrame(mrows)
MT.to_csv(os.path.join(OUT_DIR, "entries_exits_monthly.csv"), index=False)

# by analysis system (DAP / CAA ...)
srcx = (ex.groupBy("src").agg(F.count(F.lit(1)).alias("left"), F.avg(F.col("close").isNotNull().cast("int")).alias("wclose"),
                              F.sum("obs").alias("obs"), F.sum(F.col("obs") * F.col("back")).alias("back")).toPandas())
srca = pres.filter(F.col("i") < LAST).groupBy("src").agg(F.count(F.lit(1)).alias("am")).toPandas()
SR = srca.merge(srcx, on="src", how="left").fillna(0).sort_values("am", ascending=False)
SR = pd.DataFrame({"system": SR["src"], "account-months": SR["am"].map(lambda v: f"{int(v):,}"),
                   "left per month": [pct(a, b) for a, b in zip(SR["left"], SR["am"])],
                   "leavers with close date": SR["wclose"].map(fpct),
                   f"leavers back within {L}": [pct(a, b) for a, b in zip(SR["back"], SR["obs"])]})

# close dates vs the analysis month, and whether the account then leaves
ym0 = (F.col("time_key") / 100).cast("long")
cyc = ym0
for _ in range(SHIFT):
    cyc = F.when(cyc % 100 == 1, cyc - 89).otherwise(cyc - 1)
cstart = F.to_date(F.concat(cyc.cast("string"), F.lit("01")), "yyyyMMdd")
cl = (pres.filter(F.col("close").isNotNull() & (F.col("i") < LAST))
          .withColumn("when", F.when(F.col("close") < cstart, "before analysis month")
                               .when(F.col("close") <= F.last_day(cstart), "in analysis month")
                               .otherwise("after analysis month"))
          .withColumn("leaves", (F.col("nxt").isNull() | (F.col("nxt") > F.col("i") + 1)).cast("int"))
          .groupBy("when").agg(F.count(F.lit(1)).alias("rows"), F.avg("leaves").alias("leaves")).toPandas())
tot_cl = cl["rows"].sum()
CL = pd.DataFrame({"close date": cl["when"], "rows": cl["rows"].map(lambda v: f"{int(v):,}"),
                   "share": [pct(v, tot_cl) for v in cl["rows"]], "account gone next month": cl["leaves"].map(fpct)})

show([f"<h3>New and closed accounts — key {esc(' + '.join(KEY))}</h3>",
      f"<p class='note'>Left = in the snapshot before, not in this one. Returned = seen earlier in the window but not last month. "
      f"Back within {L} = reappears after at most {L} missing months (only leavers with {L} months still observable). "
      f"Balance = avg_investable_bal of the leavers in their last month, as a share of all accounts that month. "
      f"Customers = power ids; a customer leaves when all its accounts are gone. Analysis month = snapshot month minus {SHIFT}.</p>",
      tbl(MT), "<h4>By analysis system</h4>", tbl(SR),
      "<h4>Close dates (rows with a real acct_close_dt) vs analysis month</h4>", tbl(CL),
      f"<p class='note'>Saved: {esc(os.path.abspath(OUT_DIR))}</p>"])
